# 04. Feature Engineering & Leakage Prevention Pipeline
## PKL Analytics & Intelligence Platform
**Role:** Senior Data Scientist / ML Engineer  
**Dataset:** Pro Kabaddi League (Seasons 1–7)

---

### Objectives
1. **Target Formulation:** Frame the match outcome prediction as a canonical binary classification problem ($Y = 1$ if Team A wins, $0$ if Team B wins/ties).
2. **Leakage Prevention Contract:** Ensure all features are strictly derived from historical fixtures timestamped **before** the match start whistle.
3. **Domain Feature Construction:**
   - Pre-match rolling win percentages and point differentials.
   - Recent 5-match form indicators (momentum).
   - Raid and tackle efficiency rates.
   - Head-to-Head historical dominance.
   - Rest days and schedule fatigue.
   - Home venue familiarity.
4. **Temporal Partitioning:** Split data chronologically into:
   - **Training Set:** Seasons 1–5 (2014–2017)
   - **Validation Set:** Season 6 (2018–2019)
   - **Holdout Test Set:** Season 7 (2019)


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root in sys.path
sys.path.insert(0, os.path.abspath(".."))

from src.features.build_features import build_match_features, get_feature_columns, get_train_val_test_splits


### 1. Build and Inspect the Chronological Feature Matrix


In [ ]:
# Generate or load features from processed directory
df_feats = build_match_features(processed_dir="../data/processed")
feature_cols = get_feature_columns()

print(f"Total Matches Processed: {len(df_feats)}")
print(f"Total Engineered Columns: {df_feats.shape[1]}")
print(f"Vetted Predictive Feature Set ({len(feature_cols)} features):")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:02d}. {col}")


### 2. Verify Canonical Target Balance and Differential Distributions


In [ ]:
# Display sample feature vectors
cols_to_preview = [
    "match_id", "match_date", "season_id", "team_a_name", "team_b_name",
    "diff_prior_win_pct", "diff_l5_differential", "diff_raid_rate", "diff_tackle_rate",
    "target_team_a_win"
]
df_feats[cols_to_preview].head(10)


### 3. Chronological Temporal Partitioning
We avoid random K-fold CV because random shuffling leaks future meta-strategies, tactical meta-shifts, and squad aging into past evaluations.  
We enforce a strict temporal walk-forward split:


In [ ]:
train_df, val_df, test_df = get_train_val_test_splits(df_feats, exclude_ties=True)

print("=" * 60)
print(f"TRAIN SET (Seasons 1-5): {len(train_df)} matches | Target Mean: {train_df['target_team_a_win'].mean():.3f}")
print(f"VAL SET   (Season 6):    {len(val_df)} matches | Target Mean: {val_df['target_team_a_win'].mean():.3f}")
print(f"TEST SET  (Season 7):    {len(test_df)} matches | Target Mean: {test_df['target_team_a_win'].mean():.3f}")
print("=" * 60)


### 4. Correlation Analysis of Pre-Match Differential Features with Match Victory


In [ ]:
# Compute Pearson correlation with binary outcome
diff_features = [c for c in feature_cols if c.startswith("diff_") or c.startswith("h2h_")]
corrs = train_df[diff_features + ["target_team_a_win"]].corr()["target_team_a_win"].drop("target_team_a_win").sort_values(ascending=False)

plt.figure(figsize=(10, 6), dpi=150)
sns.barplot(x=corrs.values, y=corrs.index, palette="vlag")
plt.title("Pre-Match Feature Correlation with Match Victory (Training Set: S1-S5)", fontsize=12, fontweight="bold")
plt.xlabel("Pearson Correlation Coefficient")
plt.axvline(0, color="black", linestyle="--", lw=1)
plt.tight_layout()
plt.show()
